# Scoring a normalization

Normalization is the one preprocessing stage where metrics have been usable for
a while, and also the one where a single number is most misleading: every
transform that flattens per-cell variation scores well on *removal*, including
the ones that flatten the biology with it.

This page scores three transforms on the same data and shows why removal has to
be read against preservation.

## Data with two things going on

Cells belong to one of two types, which differ in marker composition. On top of
that, each cell has a technical scaling factor — detector gain, exposure, how
much of the cell the mask caught — that normalization should remove and that
has nothing to do with type.

In [1]:
import anndata as ad
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
markers = ["CD45", "ECAD", "SMA", "DNA_1"]
n_cells = 600

# Biology: two types with different marker composition.
is_immune = rng.random(n_cells) < 0.5
base = np.where(
    is_immune[:, None],
    np.array([40.0, 4.0, 6.0, 30.0]),
    np.array([4.0, 45.0, 8.0, 30.0]),
)
base = base * rng.lognormal(0.0, 0.25, size=(n_cells, len(markers)))

# Technical: a per-cell scaling factor spanning more than an order of magnitude.
depth = rng.lognormal(0.0, 0.8, n_cells)[:, None]

counts = (base * depth).astype(np.float32)

adata = ad.AnnData(
    X=counts.copy(),
    obs=pd.DataFrame(
        {
            "cell_type": pd.Categorical(np.where(is_immune, "immune", "epithelial")),
            "area": (120 * depth.ravel()).round(),
        },
        index=[str(i + 1) for i in range(n_cells)],
    ),
)
adata.var_names = markers
adata.layers["counts"] = counts.copy()
adata

AnnData object with n_obs × n_vars = 600 × 4
    obs: 'cell_type', 'area'
    layers: None (.X), 'counts'

## Three transforms

One that does nothing, one that is reasonable, and one that overdoes it.

In [2]:
totals = counts.sum(axis=1, keepdims=True)
good = np.arcsinh(counts / totals * np.median(totals) / 5.0)

# A caricature of over-correction: shrink every cell most of the way to the
# panel mean, then add a little noise. Nothing sensible does exactly this, but
# aggressive batch correction fails in this direction, and the caricature makes
# the failure easy to see.
panel_mean = good.mean(axis=0, keepdims=True)
over = panel_mean + 0.10 * (good - panel_mean) + rng.normal(0.0, 0.06, good.shape)

transforms = {"none": counts.copy(), "total + arcsinh": good, "over-corrected": over}
{k: float(np.std(v)) for k, v in transforms.items()}

{'none': 45.01640701293945,
 'total + arcsinh': 0.9035190939903259,
 'over-corrected': 0.49276100955421487}

## Scoring them

Two questions, not one: did the technical variation go, and did the biology
survive?

In [3]:
import sprotmetrics as spm

rows = {}
for name, matrix in transforms.items():
    scored = adata.copy()
    scored.layers["normalized"] = matrix.astype(np.float32)

    diagnostics = spm.normalization_diagnostics(scored)
    rows[name] = {
        # Removal: how much of the per-cell total spread survived.
        "total_cv_ratio": diagnostics["total_cv_ratio_norm_over_raw"],
        # Removal: residual dependence on the technical covariate.
        "area_r2": float(np.mean(spm.variance_explained_by_covariate(scored, layer="normalized"))),
        # Preservation: are the two cell types still apart?
        "silhouette": spm.silhouette_cell_type(scored, layer="normalized"),
        # Preservation: is the panel still contributing broadly?
        "variance_evenness": spm.marker_variance_evenness(scored, layer="normalized"),
    }

pd.DataFrame(rows).T.round(3)

,total_cv_ratio,area_r2,silhouette,variance_evenness
none,1.000,0.623,0.308,0.827
total + arcsinh,0.038,0.003,0.836,0.626
over-corrected,0.017,0.002,0.484,0.870


Read the first two columns and **over-correction wins**. It drives the per-cell
spread further down than the sensible transform does — a lower `total_cv_ratio`,
which is the direction the catalogue records — and removes the area dependence
just as completely.

Read the third column and it falls apart. The two cell types, which the raw data
separates at 0.31 and the sensible transform improves to 0.84, drop back to
roughly half that. The transform removed the technical variation by removing
most of the variation, and a metric that only measures removal cannot tell the
difference.

This is why `total_cv_ratio` is not a score. It is directional and genuinely
informative, but *lower is better* holds only while something else is holding
the biology in place.

In [4]:
catalogue = spm.list_metrics()
catalogue.loc[
    ["total_cv_ratio_norm_over_raw", "silhouette_cell_type", "marker_variance_evenness"],
    ["direction", "strength", "task"],
]

,direction,strength,task
metric_id,,,
total_cv_ratio_norm_over_raw,lower is better,strong,normalization
silhouette_cell_type,higher is better,strong,normalization
marker_variance_evenness,higher is better,strong,normalization


## What the package will not do for you

There is no composite normalization score here, and that is deliberate. Any
weighted average of removal and preservation encodes a judgement about how much
biology you are willing to trade for how much batch correction, and that
judgement belongs to the analysis, not to the metric.

What the catalogue does is make the trade visible, by refusing to collapse it.

## Next

- [Comparing two segmentations](comparing-segmentations) — the same argument
  for the segmentation axis.
- [Metric reference](../metrics) — every metric, with the mechanism drawn out.